# Proyecto #1: Biodiversity at Scale
## Partes 3 y 4: Arquitecturas CNN Modernas y Optimización
**Maestría de Investigación en IA - UTEC Posgrado**  
**Curso**: Aprendizaje Profundo – Práctica  
**Profesor**: Dra. Aurea Soriano-Vargas

---
### Objetivos de esta etapa:
1. Seleccionar y comparar al menos dos arquitecturas convolucionales modernas bajo condiciones experimentales controladas.
   - **Arquitectura A**: `ResNet-18 / ResNet-50` (Redes residuales con skip-connections).
   - **Arquitectura B**: `ConvNeXt-Tiny` (o `MiniINatCNN`), convoluciones profundas inspiradas en ViTs.
2. Comparar experimentalmente dos optimizadores clave (**SGD con Momentum** vs. **AdamW**) evaluando:
   - Velocidad de convergencia, comportamiento de la pérdida, estabilidad y capacidad de generalización.
3. Registrar los experimentos **E2, E3, E4 y E5** en el Tracker oficial.


In [ ]:
import sys
from pathlib import Path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))

import torch
from src.utils.seed import seed_everything
from src.data.dataset import SyntheticINatDataset
from src.data.dataloader import build_dataloaders
from src.models.factory import build_model, count_parameters
from src.training.losses import build_criterion
from src.training.optimizers import build_optimizer
from src.training.trainer import Trainer
from src.utils.tracking import ExperimentTracker
from src.utils.visualization import plot_comparison_curves

SEED = 42
seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tracker = ExperimentTracker(log_dir=str(ROOT_DIR / "logs"))
print(f"Device: {device}")


### 1. Carga de Datos y Configuración Experimental Común
Garantizamos condiciones estrictamente idénticas (mismo tamaño de imagen $128 \times 128$, mismo batch size $32$, misma cantidad de épocas).


In [ ]:
N_CLASSES = 50
BATCH_SIZE = 32
EPOCHS = 5
IMG_SIZE = 128

train_ds = SyntheticINatDataset(num_samples=50 * 35, num_classes=N_CLASSES, img_size=IMG_SIZE, seed=SEED)
val_ds = SyntheticINatDataset(num_samples=50 * 10, num_classes=N_CLASSES, img_size=IMG_SIZE, seed=SEED+1)

train_loader, val_loader = build_dataloaders(train_ds, val_ds, batch_size=BATCH_SIZE, num_workers=2, seed=SEED)


### 2. Comparación de Arquitecturas CNN (E2: CNN A vs E3: CNN B)
- **CNN A**: `ResNet-18` (conexiones residuales $x + F(x)$).
- **CNN B**: `MiniINatCNN` (o `ConvNeXt-Tiny`), convoluciones jerárquicas directas.


In [ ]:
criterion = build_criterion("cross_entropy")

# CNN A: ResNet-18
model_a = build_model("resnet18", num_classes=N_CLASSES, pretrained=False)
opt_a = build_optimizer(model_a, opt_type="adamw", lr=1e-3)
trainer_a = Trainer(model_a, criterion, opt_a, device=device, use_amp=True)
hist_a, best_a, vram_a, time_a = trainer_a.fit(train_loader, val_loader, epochs=EPOCHS, verbose=True)

_, _, total_m_a, _ = count_parameters(model_a)
tracker.log_experiment("E2", "ResNet-18", "AdamW", "BN", "Standard", "No", "No", best_a, time_a, vram_a, total_m_a)

# CNN B: MiniINatCNN
model_b = build_model("cnn_custom", num_classes=N_CLASSES, use_batchnorm=True, dropout_rate=0.2)
opt_b = build_optimizer(model_b, opt_type="adamw", lr=1e-3)
trainer_b = Trainer(model_b, criterion, opt_b, device=device, use_amp=True)
hist_b, best_b, vram_b, time_b = trainer_b.fit(train_loader, val_loader, epochs=EPOCHS, verbose=True)

_, _, total_m_b, _ = count_parameters(model_b)
tracker.log_experiment("E3", "MiniINatCNN", "AdamW", "BN+Dropout", "Standard", "No", "No", best_b, time_b, vram_b, total_m_b)


### 3. Comparación de Optimizadores (E4: SGD+Momentum vs E5: AdamW)
Evaluamos el impacto del optimizador fijando la arquitectura `ResNet-18`:
- **SGD con Momentum**: $m=0.9$, $lr=10^{-2}$, regularización $L_2$ integrada.
- **AdamW**: Tasas adaptativas por coordenada con weight decay desacoplado ($lr=10^{-3}$).


In [ ]:
# E4: ResNet-18 con SGD + Momentum
model_sgd = build_model("resnet18", num_classes=N_CLASSES, pretrained=False)
opt_sgd = build_optimizer(model_sgd, opt_type="sgd_momentum", lr=1e-2, weight_decay=1e-4)
trainer_sgd = Trainer(model_sgd, criterion, opt_sgd, device=device, use_amp=True)
hist_sgd, best_sgd, vram_sgd, time_sgd = trainer_sgd.fit(train_loader, val_loader, epochs=EPOCHS, verbose=True)

tracker.log_experiment("E4", "ResNet-18", "SGD+Momentum (lr=1e-2)", "BN", "Standard", "No", "No", best_sgd, time_sgd, vram_sgd, total_m_a)

# E5: ResNet-18 con AdamW
model_adamw = build_model("resnet18", num_classes=N_CLASSES, pretrained=False)
opt_adamw = build_optimizer(model_adamw, opt_type="adamw", lr=1e-3, weight_decay=1e-4)
trainer_adamw = Trainer(model_adamw, criterion, opt_adamw, device=device, use_amp=True)
hist_adamw, best_adamw, vram_adamw, time_adamw = trainer_adamw.fit(train_loader, val_loader, epochs=EPOCHS, verbose=True)

tracker.log_experiment("E5", "ResNet-18", "AdamW (lr=1e-3)", "BN", "Standard", "No", "No", best_adamw, time_adamw, vram_adamw, total_m_a)


### 4. Curvas Comparativas de Dinámica de Aprendizaje


In [ ]:
histories_opts = {
    "SGD + Momentum": hist_sgd,
    "AdamW": hist_adamw
}
plot_comparison_curves(
    histories_opts,
    metric="val_loss",
    ylabel="Validation Loss",
    title="Dinámica de Pérdida: SGD+Momentum vs. AdamW"
)

plot_comparison_curves(
    histories_opts,
    metric="val_acc",
    ylabel="Validation Accuracy (%)",
    title="Convergencia de Precisión: SGD+Momentum vs. AdamW"
)


### 5. Discusión Científica sobre Optimización y Arquitecturas
1. **AdamW** muestra convergencia inicial acelerada gracias a momentos adaptativos de primer y segundo orden ($m_t, v_t$).
2. **SGD con Momentum**, aunque requiere una selección más cuidadosa del learning rate, suele encontrar mínimos más planos (*flat minima*) con mejor capacidad de generalización en etapas avanzadas de entrenamiento.
